In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy
from scipy.stats import ttest_ind
import seaborn as sns
from math import sqrt
import re
from matplotlib.ticker import ScalarFormatter
import matplotlib.ticker as mticker
from matplotlib.colors import to_rgb, to_hex

cell_types = ["Exc", "Pvalb", "Sst", "Vip", "Htr3a"]
colors = [
    "tab:red",
    "tab:blue",
    "tab:green",
    "tab:purple",
    "tab:purple",
]
layer_colors = [
    "tab:red",
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:purple",
]
layers = ["L1", "L2/3", "L4", "L5", "L6"]

from collections import OrderedDict

Perturb_dict = OrderedDict()
Perturb_dict["no perturb"] = ""

In [ ]:
net_name = "core"  # change to "core_<variant>" for a model variant
mua_power_all = pd.read_csv(
    f"{net_name}/metrics/csv_files_dg_temporal_f_memory_efficient/dg_all_dirs_temporal_f_mua_power.csv"
)

In [ ]:
def str2list(str_time):
    str_time = str_time[1:-1]
    array_list = str_time.strip(",").split(", ")
    return np.stack([float(ii) for ii in array_list])

mua_power_all["power"] = mua_power_all["power"].apply(str2list)
mua_power_all["f"] = mua_power_all["f"].apply(str2list)

In [ ]:
# get power_df, where the results are averaged across 10 trials
power_df = mua_power_all.groupby(["condition", "layer", "cell", "temporal_f"]).agg(
    {
        "power": list,
        "f": list,
    }
)
power_df.reset_index(inplace=True)
power_df["mean"] = power_df["power"].apply(np.mean, axis=0)
power_df["std"] = power_df["power"].apply(np.std, axis=0)
power_df["f"] = power_df["f"].apply(np.mean, axis=0)

In [ ]:
layer_mua_power_all = pd.read_csv(f"{net_name}/metrics/csv_files_dg_temporal_f_memory_efficient/dg_all_dirs_temporal_f_mua_power_layer.csv")

layer_mua_power_all["power"] = layer_mua_power_all["power"].apply(str2list)
layer_mua_power_all["f"] = layer_mua_power_all["f"].apply(str2list)

In [ ]:
# get power_df, where the results are averaged across 10 trials
laminar_power_df = layer_mua_power_all.groupby(["layer", "condition", "temporal_f"]).agg(
    {
        "power": list,
        "f": list,
    }
)
laminar_power_df.reset_index(inplace=True)
laminar_power_df["mean"] = laminar_power_df["power"].apply(np.mean, axis=0)
laminar_power_df["std"] = laminar_power_df["power"].apply(np.std, axis=0)
laminar_power_df["f"] = laminar_power_df["f"].apply(np.mean, axis=0)
laminar_power_df.drop(columns=["power"], inplace=True)

mean_power_cond = laminar_power_df.groupby(["condition", "temporal_f"]).agg(
    {
        "mean": list,
    }
)
mean_power_cond["max"] = mean_power_cond["mean"].apply(
    lambda row: np.max(np.vstack(np.asanyarray(row)), axis=0)
)
mean_power_cond.rename(columns={"index": "condition"})
mean_power_cond.reset_index(inplace=True)


def get_norm_power(row):
    max_values = mean_power_cond.loc[
        mean_power_cond["condition"].isin([row["condition"]]).values
        & mean_power_cond["temporal_f"].isin([row["temporal_f"]]).values,
        "max",
    ].values[0]
    # print(row["temporal_f"])
    norm_power = row["mean"] / max_values
    # print(np.max(norm_power))
    return norm_power


laminar_power_df["norm power"] = laminar_power_df.apply(
    lambda row: get_norm_power(row), axis=1
)

In [ ]:
# color_list = [
#     "tab:red",
#     "tab:blue",
#     "tab:green",
#     "tab:purple",
# ]


# def generate_tones(base_color, num_tones=5, lightness_factor=0.2):
#     """Generates lighter tones of a given base color."""
#     rgb = to_rgb(base_color)
#     tones = []
#     for i in range(num_tones):
#         # Linearly interpolate towards white (1,1,1)
#         # The lightness_factor determines how much lighter each step is
#         new_rgb = tuple(min(1.0, c + i * lightness_factor) for c in rgb)
#         tones.append(to_hex(new_rgb))
#     return tones


# all_tones = {}
# for color_name in color_list:
#     all_tones[color_name] = generate_tones(
#         color_name, num_tones=6, lightness_factor=0.1
#     )[::-1]

In [ ]:

from matplotlib.lines import Line2D

selected_cells = ["Exc", "Pvalb", "Sst", "Vip"]

fig, axes = plt.subplots(
    ncols=5, nrows=5, sharex=True, figsize=(9.5, 6), dpi=300
)  # , sharey="col"
fig.patch.set_facecolor("none")
f = power_df.loc[0, "f"]

# colors = plt.colormaps["Accent"](np.linspace(0, 1, 8))
# colors = plt.colormaps["viridis_r"](np.linspace(0, 1, 8))
colors = np.delete(plt.colormaps["magma_r"](np.linspace(0, 1, 9)), 8, axis=0) # 

temporal_fs = [1, 2, 4, 8, 15, 30]

for jj, cell in enumerate(selected_cells):
    data = power_df.query(
        'condition == "no perturb"' + " & " + 'cell == "' + cell + '"'
    )
    print("cell {} layer_count {}".format(cell, data.layer.value_counts()))
    data.reset_index(inplace=True)
    # Hide first row axis
    axes[0, jj].get_xaxis().set_visible(False)
    axes[0, jj].get_yaxis().set_visible(False)
    axes[0, jj].set_axis_off()

    for ii, layer in enumerate(layers[1:]):
        ax = axes[ii + 1, jj]
        for ci, temporal_f in enumerate(temporal_fs):
            m = data.loc[
                (data["layer"] == layer).values & (data["temporal_f"] == temporal_f).values,
                "mean",
            ].values
            s = data.loc[
                (data["layer"] == layer).values & (data["temporal_f"] == temporal_f).values,
                "std",
            ].values / sqrt(
                10
            )  # standard error
            m, s = m[0], s[0]
            ax.plot(
                f[1:],
                m[1:],
                color=colors[ci],
                linewidth=1,
                label="temporal_f: {}".format(temporal_f),
            )
            ax.fill_between(
                f[1:],
                m[1:] - s[1:],
                m[1:] + s[1:],
                facecolor=colors[ci],
                alpha=0.4,
            )

            sns.despine()  # Removes top and right spines by default
            ax.spines["left"].set_position(("outward", 3))
            ax.spines["bottom"].set_position(("outward", 3))

            ax.set_xlim([4, 100])

            ax.set_xticks([4, 20, 40, 60, 80, 100])
            ax.set_xticklabels([4, 20, 40, 60, 80, 100], fontsize=8)
            ax.set_xticks([10, 30, 50, 70, 90], minor=True)
            ax.tick_params(axis='x', which='major', length=5)
            ax.tick_params(axis='x', which='minor', length=3)

            formatter = ScalarFormatter(useMathText=True)
            formatter.set_powerlimits((-3, 3))
            ax.yaxis.set_major_formatter(formatter)
            ax.ticklabel_format(axis="y", style="scientific", scilimits=(-3, 3))
            ax.tick_params(axis='y', labelsize=8)

            ax.text(
                1,
                0.95,
                cell + " " + layer,
                horizontalalignment="right",
                verticalalignment="top",
                fontsize=10,
                alpha=1,
                transform=ax.transAxes,
                fontweight="bold",
            )

data = laminar_power_df.query('condition == "no perturb"')

jj += 1
# Hide first row axis
axes[0, jj].get_xaxis().set_visible(False)
axes[0, jj].get_yaxis().set_visible(False)
axes[0, jj].set_axis_off()

for ii, layer in enumerate(layers[1:]):
    ax = axes[ii + 1, jj]
    for ci, temporal_f in enumerate(temporal_fs):
        m = data.loc[
            (data["layer"] == layer).values & (data["temporal_f"] == temporal_f).values,
            "mean",
        ].values
        s = data.loc[
            (data["layer"] == layer).values & (data["temporal_f"] == temporal_f).values,
            "std",
        ].values / sqrt(
            10
        )  # standard error
        m, s = m[0], s[0]
        ax.plot(
            f[1:],
            m[1:],
            color=colors[ci],
            linewidth=1,
            label="temporal_f: {}".format(temporal_f),
        )
        ax.fill_between(
            f[1:], m[1:] - s[1:], m[1:] + s[1:], facecolor=colors[ci], alpha=0.4
        )

        sns.despine()
        ax.spines["left"].set_position(("outward", 3))
        ax.spines["bottom"].set_position(("outward", 3))

        ax.set_xlim([4, 100])

        ax.set_xticks([4, 20, 40, 60, 80, 100])
        ax.set_xticklabels([4, 20, 40, 60, 80, 100], fontsize=8)
        ax.set_xticks([10, 30, 50, 70, 90], minor=True)
        ax.tick_params(axis='x', which='major', length=5)
        ax.tick_params(axis='x', which='minor', length=3)

        formatter = ScalarFormatter(useMathText=True)
        formatter.set_powerlimits((-3, 3))
        ax.yaxis.set_major_formatter(formatter)
        ax.ticklabel_format(axis="y", style="scientific", scilimits=(-3, 3))
        ax.tick_params(axis='y', labelsize=8)

        ax.text(
            1,
            0.95,
            layer,
            horizontalalignment="right",
            verticalalignment="top",
            fontsize=10,
            alpha=1,
            transform=ax.transAxes,
            fontweight="bold",
        )

# Add legend spanning the full figure width in the hidden top row
legend_elements = [
    Line2D([0], [0], color=colors[ci], linewidth=1.5, label="temporal_f: {} Hz".format(temporal_f))
    for ci, temporal_f in enumerate(temporal_fs)
]
fig.legend(
    handles=legend_elements,
    loc="upper center",
    ncol=6,
    fontsize=7,
    frameon=False,
    bbox_to_anchor=(0.5, 0.87),
)

fig.supxlabel("Frequency [Hz]", fontsize=10, y=0.035)
fig.supylabel("Power [(spks/s)²]", fontsize=10)
fig.tight_layout()
fig.savefig(
    "paper_figs/dg_temporal_fs_power_spectrum_Jul17_2026.pdf",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)


In [ ]:
selected_cells = ["Exc", "Pvalb", "Sst", "Vip"]

fig, axes = plt.subplots(
    ncols=5, nrows=5, sharex=True, figsize=(9.5, 6), dpi=300
)  # , sharey="col"
fig.patch.set_facecolor("none")
f = power_df.loc[0, "f"]

colors = plt.colormaps["Dark2"](np.linspace(0, 1, 8))

for jj, cell in enumerate(selected_cells):
    data = power_df.query(
        'condition == "no perturb"' + " & " + 'cell == "' + cell + '"'
    )
    print("cell {} layer_count {}".format(cell, data.layer.value_counts()))
    data.reset_index(inplace=True)
    # Hide first row axis
    axes[0, jj].get_xaxis().set_visible(False)
    axes[0, jj].get_yaxis().set_visible(False)

    for ii, layer in enumerate(layers[1:]):
        ax = axes[ii + 1, jj]
        for ci, temporal_f in enumerate([1, 2]):
            m = data.loc[
                (data["layer"] == layer).values & (data["temporal_f"] == temporal_f).values,
                "mean",
            ].values
            s = data.loc[
                (data["layer"] == layer).values & (data["temporal_f"] == temporal_f).values,
                "std",
            ].values / sqrt(
                10
            )  # standard error
            # if layer == "L1":
            #     # ax.set_visible(False)
            #     continue
            m, s = m[0], s[0]
            ax.plot(
                f[1:],
                m[1:],
                color=colors[ci],
                linewidth=1,
                label="temporal_f: {}".format(temporal_f),
            )
            # ax.axvline(temporal_f, color="lightgrey", linestyle="--", alpha=0.5)
            ax.fill_between(
                f[1:],
                m[1:] - s[1:],
                m[1:] + s[1:],
                facecolor=colors[ci],
                alpha=0.4,
            )

            sns.despine()  # Removes top and right spines by default
            # Position the left spine outward by 10 points
            ax.spines["left"].set_position(("outward", 3))
            # Position the bottom spine outward by 10 points
            ax.spines["bottom"].set_position(("outward", 3))

            ax.set_xlim([4, 100])

            ax.set_xticks([4, 20, 40, 60, 80, 100])
            ax.set_xticklabels([4, 20, 40, 60, 80, 100], fontsize=8)
            ax.set_xticks([10, 30, 50, 70, 90], minor=True)
            ax.tick_params(axis='x', which='major', length=5)
            ax.tick_params(axis='x', which='minor', length=3)

            # Format y-axis with scientific notation for large values
            formatter = ScalarFormatter(useMathText=True)
            formatter.set_powerlimits(
                (-3, 3)
            )  # Use scientific notation for values with >5 digits
            ax.yaxis.set_major_formatter(formatter)
            ax.ticklabel_format(axis="y", style="scientific", scilimits=(-3, 3))
            ax.tick_params(axis='y', labelsize=8)

            # # yticklabels = ax.get_yticklabels()
            # ax.set_yticks([0, 0.025, 0.05])
            # ax.set_yticklabels([0, 0.025, 0.05], fontsize=7)
            # ylim = ax.get_ylim()
            ax.text(
                1,
                0.95,
                cell + " " + layer,
                horizontalalignment="right",
                verticalalignment="top",
                fontsize=10,
                alpha=1,
                transform=ax.transAxes,
                fontweight="bold",
            )

        if layer == "L2/3":
            # Place legend in the row above (index ii-1, since L2/3 is at ii=0 for layers[1:])
            legend_ax = axes[0, jj]  # First row for this column
            legend_ax.legend(
                handles=ax.get_legend_handles_labels()[0],
                labels=ax.get_legend_handles_labels()[1],
                loc="center",
                bbox_to_anchor=(0.5, 0.5),
                mode=None,
                borderaxespad=0.0,
                fontsize=6,
            )

data = laminar_power_df.query('condition == "no perturb"')
# colors = generate_tones("grey", num_tones=6, lightness_factor=0.1)[::-1]
# colors = plt.colormaps["copper_r"](np.linspace(0, 1, 6))

jj += 1
# Hide first row axis
axes[0, jj].get_xaxis().set_visible(False)
axes[0, jj].get_yaxis().set_visible(False)
for ii, layer in enumerate(layers[1:]):
    ax = axes[ii + 1, jj]
    for ci, temporal_f in enumerate([1, 2]):
        m = data.loc[
            (data["layer"] == layer).values & (data["temporal_f"] == temporal_f).values,
            "mean",
        ].values
        s = data.loc[
            (data["layer"] == layer).values & (data["temporal_f"] == temporal_f).values,
            "std",
        ].values / sqrt(
            10
        )  # standard error
        # if layer == "L1":
        #     # ax.set_visible(False)
        #     continue
        m, s = m[0], s[0]
        ax.plot(
            f[1:],
            m[1:],
            color=colors[ci],
            linewidth=1,
            label="temporal_f: {}".format(temporal_f),
        )
        # ax.axvline(temporal_f, color="lightgrey", linestyle="--", alpha=0.5)
        ax.fill_between(
            f[1:], m[1:] - s[1:], m[1:] + s[1:], facecolor=colors[ci], alpha=0.4
        )

        sns.despine()  # Removes top and right spines by default
        # Position the left spine outward by 10 points
        ax.spines["left"].set_position(("outward", 3))
        # Position the bottom spine outward by 10 points
        ax.spines["bottom"].set_position(("outward", 3))

        ax.set_xlim([4, 100])

        ax.set_xticks([4, 20, 40, 60, 80, 100])
        ax.set_xticklabels([4, 20, 40, 60, 80, 100], fontsize=8)
        ax.set_xticks([10, 30, 50, 70, 90], minor=True)
        ax.tick_params(axis='x', which='major', length=5)
        ax.tick_params(axis='x', which='minor', length=3)

        # Format y-axis with scientific notation for large values
        formatter = ScalarFormatter(useMathText=True)
        formatter.set_powerlimits(
            (-3, 3)
        )  # Use scientific notation for values with >5 digits
        ax.yaxis.set_major_formatter(formatter)
        ax.ticklabel_format(axis="y", style="scientific", scilimits=(-3, 3))
        ax.tick_params(axis='y', labelsize=8)

        # # yticklabels = ax.get_yticklabels()
        # ax.set_yticks([0, 0.025, 0.05])
        # ax.set_yticklabels([0, 0.025, 0.05], fontsize=7)
        # ylim = ax.get_ylim()
        ax.text(
            1,
            0.95,
            layer,
            horizontalalignment="right",
            verticalalignment="top",
            fontsize=10,
            alpha=1,
            transform=ax.transAxes,
            fontweight="bold",
        )

    if layer == "L2/3":
        # Place legend in the row above (index ii-1, since L2/3 is at ii=0 for layers[1:])
        legend_ax = axes[0, jj]  # First row for this column
        legend_ax.legend(
            handles=ax.get_legend_handles_labels()[0],
            labels=ax.get_legend_handles_labels()[1],
            loc="center",
            bbox_to_anchor=(0.5, 0.5),
            mode=None,
            borderaxespad=0.0,
            fontsize=6,
        )

fig.supxlabel("Frequency [Hz]", fontsize=10)
fig.supylabel("Power [(spks/s)²]", fontsize=10)
fig.tight_layout()
# fig.savefig(
#     "paper_figs/dg_temporal_fs_power_spectrum.pdf",
#     dpi=300,
#     bbox_inches="tight",
#     transparent=True,
# )
